In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt


def orquestrador_tabela_a1():
    print("\n" + "=" * 72)
    print("TABELA A1: DISTRIBUIÇÃO DO VOLUME DE INSCRITOS POR SITUAÇÃO")
    print("=" * 72)

    caminho_base = Path("../data/04_load/database/inscritos_final_limpo.parquet")
    pasta_saida = Path("../reports/figures/apendices")
    pasta_saida.mkdir(parents=True, exist_ok=True)

    df = carregar_base(caminho_base)
    df_tabela = preparar_tabela_a1(df)

    print("\nTabela A1:")
    print(df_tabela)

    plotar_tabela_a1_como_imagem(
        df_tabela=df_tabela,
        pasta_saida=pasta_saida
    )


def carregar_base(caminho_base):
    print("[*] Carregando base...")
    df = pd.read_parquet(str(caminho_base))
    print(f"Total de registros: {len(df):,}".replace(",", "."))
    return df


def normalizar_status(texto):
    return str(texto).strip().upper()


def preparar_tabela_a1(df):
    df = df.copy()

    mapa_status = {
        "CONTRATADA": "Contratada",
        "INSCRIÇÃO POSTERGADA": "Inscrição postergada",
        "PRE-SELECIONADO": "Pré-selecionado",
        "PRÉ-SELECIONADO": "Pré-selecionado",
        "NÃO CONTRATADO": "Não contratado",
        "NAO CONTRATADO": "Não contratado",
        "REJEITADA PELA CPSA": "Rejeitada pela CPSA",
        "OPÇÃO NÃO CONTRATADA": "Opção não contratada",
        "OPCAO NAO CONTRATADA": "Opção não contratada",
        "PARTICIPACAO CANCELADA PELO CANDIDATO": "Participação cancelada",
        "PARTICIPAÇÃO CANCELADA PELO CANDIDATO": "Participação cancelada",
        "LISTA DE ESPERA": "Lista de espera"
    }

    df["status_norm"] = df["situacao_fies"].apply(normalizar_status)
    df["status_tabela"] = df["status_norm"].map(mapa_status).fillna("Outros")

    df_agg = (
        df.groupby("status_tabela", as_index=False)
          .size()
          .rename(columns={"size": "Quantidade de inscritos"})
    )

    total = df_agg["Quantidade de inscritos"].sum()
    df_agg["%"] = df_agg["Quantidade de inscritos"] / total * 100

    # Ordena da maior para a menor frequência
    df_agg = df_agg.sort_values(
        by="Quantidade de inscritos",
        ascending=False
    ).reset_index(drop=True)

    df_agg = df_agg.rename(columns={
        "status_tabela": "Situação da inscrição"
    })

    return df_agg


def formatar_inteiro(valor):
    return f"{int(valor):,}".replace(",", ".")


def formatar_percentual(valor):
    return f"{valor:.1f}%".replace(".", ",")


def plotar_tabela_a1_como_imagem(df_tabela, pasta_saida):
    df_fmt = df_tabela.copy()

    df_fmt["Quantidade de inscritos"] = (
        df_fmt["Quantidade de inscritos"].apply(formatar_inteiro)
    )
    df_fmt["%"] = df_fmt["%"].apply(formatar_percentual)

    plt.rcParams["font.family"] = "DejaVu Sans"

    # Ajuste dinâmico de altura com base no número de linhas
    n_linhas = len(df_fmt)
    altura = max(2.2, 0.48 * (n_linhas + 1))

    fig = plt.figure(figsize=(7.2, altura), dpi=300)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.axis("off")

    tabela = ax.table(
        cellText=df_fmt.values,
        colLabels=df_fmt.columns,
        cellLoc="center",
        colLoc="center",
        bbox=[0, 0, 1, 1],   # ocupa 100% do eixo
        colWidths=[0.52, 0.30, 0.18]
    )

    tabela.auto_set_font_size(False)
    tabela.set_fontsize(10.5)

    cor_cabecalho = "#1f4e79"
    cor_linha_1 = "#eaf2f8"
    cor_linha_2 = "#ffffff"
    cor_borda = "#b7c9d6"

    for (linha, coluna), celula in tabela.get_celld().items():
        celula.set_edgecolor(cor_borda)
        celula.set_linewidth(0.8)
        celula.PAD = 0.025

        if linha == 0:
            celula.set_facecolor(cor_cabecalho)
            celula.get_text().set_color("white")
            celula.get_text().set_fontweight("bold")
            celula.get_text().set_ha("center")
            celula.get_text().set_va("center")
        else:
            celula.set_facecolor(
                cor_linha_1 if linha % 2 == 1 else cor_linha_2
            )

            if coluna == 0:
                celula.get_text().set_fontweight("bold")
                celula.get_text().set_ha("left")
            else:
                celula.get_text().set_ha("center")

            celula.get_text().set_va("center")

    caminho_png = pasta_saida / "tabela_A1_situacao_inscricao.png"
    caminho_pdf = pasta_saida / "tabela_A1_situacao_inscricao.pdf"

    fig.savefig(
        caminho_png,
        dpi=700,
        bbox_inches="tight",
        pad_inches=0.01
    )

    fig.savefig(
        caminho_pdf,
        bbox_inches="tight",
        pad_inches=0.01
    )

    plt.close(fig)

    print(f"Tabela A1 exportada para: {caminho_png}")
    print(f"Tabela A1 em PDF exportada para: {caminho_pdf}")


orquestrador_tabela_a1()


TABELA A1: DISTRIBUIÇÃO DO VOLUME DE INSCRITOS POR SITUAÇÃO
[*] Carregando base...
Total de registros: 2.197.234

Tabela A1:
    Situação da inscrição  Quantidade de inscritos          %
0          Não contratado                  1024604  46.631538
1         Lista de espera                   740698  33.710474
2              Contratada                   160369   7.298676
3    Opção não contratada                   139469   6.347481
4  Participação cancelada                   111202   5.060999
5         Pré-selecionado                    12923   0.588149
6     Rejeitada pela CPSA                     5645   0.256914
7    Inscrição postergada                     2324   0.105769
Tabela A1 exportada para: ../reports/figures/apendices/tabela_A1_situacao_inscricao.png
Tabela A1 em PDF exportada para: ../reports/figures/apendices/tabela_A1_situacao_inscricao.pdf
